In [2]:
import numpy as np
import math

In [87]:
def permn(d: int, N: int) -> np.ndarray:
    '''
    V: Indexes of the sums \\
    N: Number of sums
    
    [M, I] -> [Combination, Indexes]
    '''
    if N < 0:
        raise("Second argument should be a positive interger")
    
    V = np.arange(d+1)

    nV = V.shape[0]
    # M = np.zeros(shape=(nV, N))
    I = np.zeros(shape=(nV, N))
    # Return all permutations

    if nV == 0 or N == 0:
        # M = np.zeros(shape=(nV, N))
        I = np.zeros(shape=(nV, N))
    elif N == 1:
        # M = V.reshape((nV, 1))
        I = np.arange(nV).T
    else:
        I = np.flip(np.array(np.meshgrid(*[np.arange(nV) for i in range(N)], indexing="ij")).T.reshape(-1, N), axis=1)
        # M = V[I]

    I = I[[(ind[0] + ind[1]) == d for ind in I]]
    return I

def permutations_for_lmis(perm: np.ndarray, r: int, K = None) -> tuple[np.ndarray, np.ndarray]:
    '''
    perm: permutation of indexes \\
    r: number of rules
    '''
    indexes = []
    for comb in perm:
        indexes.append(np.array([comb]))
    	
    for i in range(r):
        for j in range(r):
            if i == j:
                continue
            for q in range(r):
                ind = np.array([
                    [i, i, q],
                    [i, j, q],
                    [j, i, q]
                ])

                indexes.append(ind)

    return indexes

def gen_coef(index: np.ndarray, coef='num', one=False):
    order = index.max()

    index_out = np.zeros(shape=(index.shape[0], ), dtype=object)

    order_fact = math.factorial(order)

    for k in range(order+1):
        new_item = dict()
        
        # index_out[k, 0] = {'alpha': index[k]}
        new_item['alpha'] = index[k]

        c_num = None
        
        if not one:
            c_num = order_fact / (math.factorial(k)*math.factorial(order - k))
        else:
            c_num = 1
            
        new_item[coef] = [c_num, new_item['alpha'].tolist()]


        index_out[k] = new_item

    return index_out

def generate_lmis():
    pass

def multiply(left_terms: np.ndarray[object], right_terms: np.ndarray[object]):
    for l_term in left_terms:
        for r_term in right_terms:
            add = True

            alpha_order = (l_term['alpha'] + r_term['alpha']).tolist()
            numerical = 1

            # check if num exists in both or in any
            if 'num' in l_term and 'num' in r_term:
                numerical = l_term['num'][0] * r_term['num'][0]
            elif 'num' in l_term:
                numerical = l_term['num'][0]
            elif 'num' in r_term:
                numerical = r_term['num'][0]

            # there is a problem with the product of matrices - better to deal with the LMIs directly

            # check if P exists in both or in any          
            newP = np.array([])
            newA = np.array([])

            if 'P' in l_term and 'P' in r_term:
                newP = np.concat([l_term['P'], r_term['P']])
            elif 'P' in l_term:
                newP = l_term['P']
            elif 'P' in r_term:
                newP = r_term['P']

            if 'A' in l_term and 'A' in r_term:
                newA = np.concat([l_term[]])

            print(alpha_order, numerical, newP)
            
            # must deal with num -> multiply with the num of the first term P or A -> if exists, else add as a num

            # for P
            #     append P
            # for A
            #     append A

            # if alrealdy existis this becomes a sum
            if len(product_result) > 0:      
                for i in range(len(product_result)):
                    if product_result[i]['alpha'] == alpha_order:
                        product_result[i]['P'].append(P)
                        add = False
                        break
            
            if add:
                new_object = dict()
                new_object['alpha'] = alpha_order
                new_object['A'] = []
                new_object['P'] = [P]

                product_result.append(new_object)
        

In [75]:
d = 2
p = 2
n_alphas = 2

d_pol = gen_coef(permn(d, n_alphas))

A_pol_T = gen_coef(permn(1, n_alphas), "A'", True)
P_pol   = gen_coef(permn(1, n_alphas), 'P', True)
A_pol   = gen_coef(permn(1, n_alphas), 'A', True)

d_pol, A_pol_T, P_pol, A_pol

(array([{'alpha': array([0, 2]), 'num': [1.0, [0, 2]]},
        {'alpha': array([1, 1]), 'num': [2.0, [1, 1]]},
        {'alpha': array([2, 0]), 'num': [1.0, [2, 0]]}], dtype=object),
 array([{'alpha': array([0, 1]), "A'": [1, [0, 1]]},
        {'alpha': array([1, 0]), "A'": [1, [1, 0]]}], dtype=object),
 array([{'alpha': array([0, 1]), 'P': [1, [0, 1]]},
        {'alpha': array([1, 0]), 'P': [1, [1, 0]]}], dtype=object),
 array([{'alpha': array([0, 1]), 'A': [1, [0, 1]]},
        {'alpha': array([1, 0]), 'A': [1, [1, 0]]}], dtype=object))

In [ ]:
generate_lmis()

In [10]:
dif_result = []
product_result = []

for d_coef in d_pol:
    for p_coef in p_pol:
        add = True

        alpha_order = (d_coef[:-1] + p_coef).tolist()
        P = (int(d_coef[-1]), p_coef.tolist())

        # seach if it already exists
        if len(product_result) > 0:      
            for i in range(len(product_result)):
                if product_result[i]['alpha'] == alpha_order:
                    product_result[i]['P'].append(P)
                    add = False
                    break
        
        if add:
            new_object = dict()
            new_object['alpha'] = alpha_order
            new_object['A'] = []
            new_object['P'] = [P]

            product_result.append(new_object)


product_result = np.array(product_result, dtype=object)

In [11]:
product_result

array([{'alpha': [0, 4], 'A': [], 'P': [(1, [0, 2])]},
       {'alpha': [1, 3], 'A': [], 'P': [(1, [1, 1]), (2, [0, 2])]},
       {'alpha': [2, 2], 'A': [], 'P': [(1, [2, 0]), (2, [1, 1]), (1, [0, 2])]},
       {'alpha': [3, 1], 'A': [], 'P': [(2, [2, 0]), (1, [1, 1])]},
       {'alpha': [4, 0], 'A': [], 'P': [(1, [2, 0])]}], dtype=object)

In [7]:
product_result[:, 0].reshape(9, 1)

ValueError: cannot reshape array of size 5 into shape (9,1)

In [312]:
find = [2, 2]


2
4
6


In [313]:
search_in

array([[list([0, 4])],
       [list([1, 3])],
       [list([2, 2])],
       [list([1, 3])],
       [list([2, 2])],
       [list([3, 1])],
       [list([2, 2])],
       [list([3, 1])],
       [list([4, 0])]], dtype=object)